# 17 - Task Diversity: Analysis

Compare legibility scores across tasks (GSM8K, SVAMP, ARC-Challenge, HellaSwag).
Does legibility vary by reasoning type?

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def get_acc(condition):
    results = []
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        r = json.loads(p.read_text())
        # For MC tasks, comparison might be string-based
        results.append(str(r["predicted_answer"]) == str(r["gold_answer"]))
    return np.mean(results) if results else None

def bootstrap_ci(condition, n=10000, seed=42):
    results = []
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        r = json.loads(p.read_text())
        results.append(str(r["predicted_answer"]) == str(r["gold_answer"]))
    if not results:
        return None, None
    arr = np.array(results, dtype=float)
    rng = np.random.RandomState(seed)
    boots = sorted([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n)])
    return boots[int(0.025*n)], boots[int(0.975*n)]

ALL_TASKS = ["gsm8k", "svamp", "arc_challenge", "hellaswag"]
TASK_LABELS = ["GSM8K", "SVAMP", "ARC-Challenge", "HellaSwag"]

In [ ]:
task_results = []

for task_name, label in zip(ALL_TASKS, TASK_LABELS):
    # GSM8K uses core conditions (no suffix)
    if task_name == "gsm8k":
        nc = get_acc("no_cot")
        sp = get_acc("self_prefill")
        cp = get_acc("cross_prefill")
        ps = get_acc("paraphrase_self")
        pc = get_acc("paraphrase_cross")
    else:
        nc = get_acc(f"no_cot_{task_name}")
        sp = get_acc(f"self_prefill_{task_name}")
        cp = get_acc(f"cross_prefill_{task_name}")
        ps = get_acc(f"paraphrase_self_{task_name}")
        pc = get_acc(f"paraphrase_cross_{task_name}")

    total_value = (sp - nc) if (sp is not None and nc is not None) else None
    L = ((pc - nc) / total_value) if (pc is not None and total_value and total_value > 0.01) else None

    row = {
        "task": task_name, "label": label,
        "no_cot": nc, "self_prefill": sp, "cross_prefill": cp,
        "paraphrase_self": ps, "paraphrase_cross": pc,
        "total_value": total_value, "L": L,
    }
    task_results.append(row)
    print(f"{label:20s} | nc={nc} sp={sp} cp={cp} ps={ps} pc={pc} | L={L}")

with open(RESULTS_DIR / "task_diversity_results.json", "w") as f:
    json.dump(task_results, f, indent=2, default=str)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

valid = [r for r in task_results if r["self_prefill"] is not None]
labels = [r["label"] for r in valid]

# Left: 2x2 conditions per task (grouped bar)
x = np.arange(len(valid))
w = 0.18
conditions_plot = [
    ("no_cot", "No COT", "#95a5a6"),
    ("self_prefill", "Self prefill", "#8e44ad"),
    ("cross_prefill", "Cross prefill", "#9b59b6"),
    ("paraphrase_self", "Para self", "#2980b9"),
    ("paraphrase_cross", "Para cross", "#27ae60"),
]

for j, (key, cond_label, color) in enumerate(conditions_plot):
    vals = [r[key] if r[key] is not None else 0 for r in valid]
    ax1.bar(x + (j - 2) * w, vals, w, label=cond_label, color=color,
            edgecolor="black", linewidth=0.3)

ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylabel("Accuracy")
ax1.set_title("Accuracy by Task and Condition", fontweight="bold")
ax1.legend(fontsize=7, ncol=2)
ax1.set_ylim(0, 1)

# Right: L per task
l_valid = [(r["label"], r["L"]) for r in valid if r["L"] is not None]
if l_valid:
    tl, tv = zip(*l_valid)
    task_colors = ["#e74c3c", "#3498db", "#f39c12", "#27ae60"]
    bars = ax2.bar(range(len(tl)), tv, color=task_colors[:len(tl)],
                   edgecolor="black", linewidth=0.5)
    for bar, val in zip(bars, tv):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{val:.3f}", ha="center", fontsize=10)
    ax2.set_xticks(range(len(tl)))
    ax2.set_xticklabels(tl)

ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)
ax2.axhline(y=0.5, color="red", linestyle="--", alpha=0.3)
ax2.set_ylabel("Legibility score (L)")
ax2.set_title("Legibility by Task Type", fontweight="bold")
ax2.set_ylim(0, 1.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "task_diversity.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Summary
print("=" * 60)
print("TASK DIVERSITY SUMMARY")
print("=" * 60)

df = pd.DataFrame(task_results)
cols = ["label", "no_cot", "self_prefill", "paraphrase_cross", "total_value", "L"]
available_cols = [c for c in cols if c in df.columns]
print(df[available_cols].to_string(index=False, float_format="{:.3f}".format))

l_vals = [r["L"] for r in task_results if r["L"] is not None]
if l_vals:
    print(f"\nLegibility across tasks:")
    print(f"  Mean L:  {np.mean(l_vals):.3f}")
    print(f"  Std L:   {np.std(l_vals):.3f}")
    print(f"  Range:   [{np.min(l_vals):.3f}, {np.max(l_vals):.3f}]")

    highest = task_results[np.argmax(l_vals)]
    lowest = task_results[np.argmin(l_vals)]
    print(f"  Most legible:  {highest['label']} (L={highest['L']:.3f})")
    print(f"  Least legible: {lowest['label']} (L={lowest['L']:.3f})")